In [2]:
# Define the query
query = """
SELECT *
FROM [dbo].[tbl_errordetails_datamodel]
where file_id > ISNULL((SELECT max(FILE_ID)
        FROM [dbo].[tbl_error_count_by_attribute_datamodel]), 0);
"""

# Execute the query
cursor.execute(query)

# Fetch all results
rows = cursor.fetchall()

# Convert result to DataFrame
columns = [column[0] for column in cursor.description]

# Convert pyodbc.Row objects to list of tuples
rows = [tuple(row) for row in rows]

# Convert result to Spark DataFrame
if rows:
    df = spark.createDataFrame(rows, schema=columns)

df.createOrReplaceTempView('vw_tbl_errordetails_datamodel')

In [3]:
def sp_error_by_attribute_and_attributegroup_datamodel():
    tbl_error_count_by_attribute_datamodel = spark.sql("""
        SELECT  
            file_id,
            attribute_group, 
            business_attribute_name, 
            IFNULL(non_confirmed, 0) AS non_confirmed,
            CASE 
                WHEN non_confirmed > 0 AND IFNULL(conformity_percent, 0) = 0 THEN 1.00 
                ELSE conformity_percent 
            END AS conformity_percent, 
            IFNULL(incomplete, 0) AS incomplete, 
            CASE 
                WHEN incomplete > 0 AND IFNULL(completeness_percent, 0) = 0 THEN 1.00 
                ELSE completeness_percent 
            END AS completeness_percent,
            IFNULL(duplicate, 0) AS duplicate,
            CASE 
                WHEN duplicate > 0 AND IFNULL(duplicity_percent, 0) = 0 THEN 1.00 
                ELSE duplicity_percent 
            END AS duplicity_percent,
            preferred_asset,
            total_rows, 
            unique_identifier,
            dimension,
            CASE 
                WHEN dimension = 'Completeness' THEN incomplete
                WHEN dimension = 'Duplicity' THEN duplicate
                ELSE non_confirmed 
            END AS totalerror 
        FROM (
            SELECT 
                file_id, 
                attribute_group, 
                business_attribute_name,
                SUM(CASE WHEN dimension = 'Conformity' THEN 1 ELSE 0 END) AS non_confirmed,
                SUM(CASE WHEN dimension = 'Conformity' THEN 1 ELSE 0 END) * 100.0 / NULLIF(COUNT(*) OVER(), 0) AS conformity_percent,
                SUM(CASE WHEN dimension = 'Completeness' THEN 1 ELSE 0 END) AS incomplete,
                SUM(CASE WHEN dimension = 'Completeness' THEN 1 ELSE 0 END) * 100.0 / NULLIF(COUNT(*) OVER(), 0) AS completeness_percent,
                SUM(CASE WHEN dimension = 'Duplicity' THEN 1 ELSE 0 END) AS duplicate,
                SUM(CASE WHEN dimension = 'Duplicity' THEN 1 ELSE 0 END) * 100.0 / NULLIF(COUNT(*) OVER(), 0) AS duplicity_percent,
                'Default' AS preferred_asset,
                COUNT(*) AS total_rows,
                CONCAT(value, '$', error_id, '$' ,dimension, '$', attribute_group, '$', business_attribute_name) AS unique_identifier,
                dimension
            FROM 
                vw_tbl_errordetails_datamodel B
            GROUP BY 
                file_id, 
                attribute_group, 
                business_attribute_name, 
                value, 
                error_id, 
                dimension
        )
    """)
    
    # Create temp view
    tbl_error_count_by_attribute_datamodel.createOrReplaceTempView('tbl_error_count_by_attribute_datamodel')
    
    # Overwrite table in the database
    tbl_error_count_by_attribute_datamodel.toPandas().to_sql(
        'tbl_error_count_by_attribute_datamodel',
        con=engine,
        if_exists='append',
        index=False
    )

In [4]:
sp_error_by_attribute_and_attributegroup_datamodel()